In [ ]:
"""
Reference  
https://www.youtube.com/watch?v=aGOyF5aU9zc : Soft Actor Critic (SAC) Code implementation -  AILinkDeepTech


Chatgpt chatbox link : https://chatgpt.com/share/67e33a52-0384-8007-9165-a2a92ccc49c5


"""

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random


stock_data = pd.read_csv("C:/Users/YEP!/Desktop/project/historical_data.csv")
print(f"Loaded {len(stock_data)} rows of stock data.")

#Data Preprocessing: Select relevant columns and normalize
stock_data['date'] = pd.to_datetime(stock_data['date'])
stock_data['close'] = stock_data['close'].astype(float)
stock_data['open'] = stock_data['open'].astype(float)
stock_data['high'] = stock_data['high'].astype(float)
stock_data['low'] = stock_data['low'].astype(float)
stock_data['volume'] = stock_data['volume'].astype(float)

#Selecting features for state
features = ['close', 'open', 'high', 'low', 'volume']


def get_state(data, window_size=10):
    state = []
    for i in range(len(data) - window_size):
        window = data.iloc[i:i+window_size][features].values  
        state.append(window.flatten())  #Flatten the window to create the state
    return np.array(state)

#Preprocess stock data into states
window_size = 10
state_data = get_state(stock_data, window_size)


print(f"State data shape: {state_data.shape}") #Check the shape of state data

#ReplayBuffer
class ReplayBuffer:
    def __init__(self, buffer_size, batch_size):
        self.memory = deque(maxlen=buffer_size)
        self.batch_size = batch_size

    def add(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def sample(self):
        experiences = random.sample(self.memory, k=self.batch_size)

        states = torch.FloatTensor([e[0] for e in experiences])
        actions = torch.FloatTensor([e[1] for e in experiences])
        rewards = torch.FloatTensor([e[2] for e in experiences])
        next_states = torch.FloatTensor([e[3] for e in experiences])
        dones = torch.FloatTensor([e[4] for e in experiences])

        return states, actions, rewards, next_states, dones


#SoftQNetwork (Q-function approximator)
class SoftQNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super(SoftQNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim + action_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)

    def forward(self, state, action):
        x = torch.cat([state, action], 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


#Policy Network (Policy approximator)
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=256, log_std_min=-20, log_std_max=2):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)

        self.mean_linear = nn.Linear(hidden_dim, action_dim)
        self.log_std_linear = nn.Linear(hidden_dim, action_dim)

        #Constraints for the log standard deviation
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))

        mean = self.mean_linear(x)
        log_std = self.log_std_linear(x)
        log_std = torch.clamp(log_std, self.log_std_min, self.log_std_max)

        return mean, log_std

    def sample(self, state):
        mean, log_std = self.forward(state)
        std = log_std.exp()

        #Reparameterization trick
        noise = torch.randn_like(mean)
        action = mean + std * noise

        #Compute log probability
        log_prob = -0.5 * ((noise ** 2) + 2 * log_std + np.log(2 * np.pi))
        log_prob = log_prob.sum(dim=1, keepdim=True)

        #Tanh squashing
        action = torch.tanh(action)
        log_prob -= torch.log(1 - action.pow(2) + 1e-6).sum(dim=1, keepdim=True)

        return action, log_prob


#SAC Agent
class SACAgent:
    def __init__(self, state_dim, action_dim, lr=3e-4, gamma=0.99, tau=0.005, target_entropy=None):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        #Initialize the networks
        self.policy_net = PolicyNetwork(state_dim, action_dim).to(self.device)
        self.policy_optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)

        self.q_net1 = SoftQNetwork(state_dim, action_dim).to(self.device)
        self.q_net2 = SoftQNetwork(state_dim, action_dim).to(self.device)
        self.q_optimizer1 = optim.Adam(self.q_net1.parameters(), lr=lr)
        self.q_optimizer2 = optim.Adam(self.q_net2.parameters(), lr=lr)

        self.target_q_net1 = SoftQNetwork(state_dim, action_dim).to(self.device)
        self.target_q_net2 = SoftQNetwork(state_dim, action_dim).to(self.device)
        self.target_q_net1.load_state_dict(self.q_net1.state_dict())
        self.target_q_net2.load_state_dict(self.q_net2.state_dict())

        self.gamma = gamma
        self.tau = tau

        # Temperature parameter
        if target_entropy is None:
            self.target_entropy = -action_dim
        else:
            self.target_entropy = target_entropy

        self.log_alpha = torch.zeros(1, requires_grad=True, device=self.device)
        self.alpha_optimizer = optim.Adam([self.log_alpha], lr=lr)

        self.replay_buffer = ReplayBuffer(buffer_size=100000, batch_size=256)

    @property
    def alpha(self):
        return self.log_alpha.exp()

    def select_action(self, state):
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            action, _ = self.policy_net.sample(state)
        return action.cpu().numpy()[0]

    def train(self):
        if len(self.replay_buffer.memory) < self.replay_buffer.batch_size:
            return

        #Sample batch from replay buffer
        states, actions, rewards, next_states, dones = self.replay_buffer.sample()
        states = states.to(self.device)
        actions = actions.to(self.device)
        rewards = rewards.to(self.device)
        next_states = next_states.to(self.device)
        dones = dones.to(self.device)

        q1_current = self.q_net1(states, actions)
        q2_current = self.q_net2(states, actions)

        #Sample next actions from policy
        next_actions, next_log_probs = self.policy_net.sample(next_states)

        next_q1 = self.target_q_net1(next_states, next_actions)
        next_q2 = self.target_q_net2(next_states, next_actions)
        next_q_target = torch.min(next_q1, next_q2) - self.alpha * next_log_probs

        #Compute target Q-values
        q_target = rewards + (1 - dones) * self.gamma * next_q_target

        #Q-network losses
        q1_loss = F.mse_loss(q1_current, q_target.detach())
        q2_loss = F.mse_loss(q2_current, q_target.detach())

        #Update Q-networks
        self.q_optimizer1.zero_grad()
        q1_loss.backward()
        self.q_optimizer1.step()

        self.q_optimizer2.zero_grad()
        q2_loss.backward()
        self.q_optimizer2.step()

        #Policy Network Loss
        sampled_actions, log_prob = self.policy_net.sample(states)
        q1_pi = self.q_net1(states, sampled_actions)
        q2_pi = self.q_net2(states, sampled_actions)
        q_pi = torch.min(q1_pi, q2_pi)

        policy_loss = (self.alpha * log_prob - q_pi).mean()

        #Update policy network
        self.policy_optimizer.zero_grad()
        policy_loss.backward()
        self.policy_optimizer.step()

        #Update Temperature parameter
        alpha_loss = -(self.log_alpha * (log_prob.detach() + self.target_entropy)).mean()
        self.alpha_optimizer.zero_grad()
        alpha_loss.backward()
        self.alpha_optimizer.step()

        #Soft Update of target networks
        self._soft_update(self.q_net1, self.target_q_net1)
        self._soft_update(self.q_net2, self.target_q_net2)

    def _soft_update(self, local_model, target_model):
        for target_param, local_param in zip(target_model.parameters(), local_model.parameters()):
            target_param.data.copy_(
                self.tau * local_param.data + (1.0 - self.tau) * target_param.data
            )

    def store_transition(self, state, action, reward, next_state, done):
        self.replay_buffer.add(state, action, reward, next_state, done)

# Training function for SAC on stock data
def train_sac_on_stock_data():
    agent = SACAgent(state_dim=state_data.shape[1], action_dim=1)

    num_episodes = 5
    max_steps = len(state_data) - window_size  #Total steps based on available data

    for episode in range(num_episodes):
        state_idx = 0
        state = state_data[state_idx]
        episode_reward = 0

        for step in range(max_steps):
            action = agent.select_action(state)
            next_state = state_data[state_idx + 1] if state_idx + 1 < len(state_data) else state
            reward = 0  
            done = state_idx + 1 >= len(state_data)

            agent.store_transition(state, action, reward, next_state, done)
            agent.train()

            state = next_state
            state_idx += 1
            episode_reward += reward

            if done:
                break

        print(f"Episode {episode+1}: Total Reward = {episode_reward}")

    return agent

#Run training
if __name__ == "__main__":
    agent = train_sac_on_stock_data()


Loaded 4066965 rows of stock data.
State data shape: (4066955, 50)


C:\Users\YEP!\AppData\Local\Temp\ipykernel_26460\3374809016.py:52: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  states = torch.FloatTensor([e[0] for e in experiences])
C:\Users\YEP!\AppData\Local\Temp\ipykernel_26460\3374809016.py:188: UserWarning: Using a target size (torch.Size([256, 256])) that is different to the input size (torch.Size([256, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  q1_loss = F.mse_loss(q1_current, q_target.detach())
C:\Users\YEP!\AppData\Local\Temp\ipykernel_26460\3374809016.py:189: UserWarning: Using a target size (torch.Size([256, 256])) that is different to the input size (torch.Size([256, 1])). This will likely lead to incorrect results 